# Name: Aman Mujahid
# Github Username: amanmujahid
# USC ID: 7092517106

## Question 1

### (a)

In [1]:
import os
import numpy as np
import pandas as pd
import re

In [2]:
root = "../data/AReM"

activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
cols = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

def read_file(path):
    rows = []
    for line in open(path):
        if line.startswith("#") or not line.strip():
            continue
        rows.append([float(x) for x in re.split(r"[,\s]+", line.strip())[:7]])
    return pd.DataFrame(rows, columns=cols)

data = []

for act in activities:
    n_files = len([f for f in os.listdir(os.path.join(root, act)) if f.endswith(".csv")])
    for i in range(1, n_files + 1):
        data.append((act, i, read_file(os.path.join(root, act, f"dataset{i}.csv"))))


print("Total number of instances:", len(data))

Total number of instances: 88


### (b)

In [3]:
train = []
test = []

for act, i, df in data:
    if act in ("bending1", "bending2"):
        n_test = 2
    else:
        n_test = 3
    if i <= n_test:
        test.append((act, i, df))
    else:
        train.append((act, i, df))

print("Number of training instances:", len(train))
print("Number of test instances:", len(test))

Number of training instances: 69
Number of test instances: 19


### c(i)

From my research, the time-domain features commonly used in time series classification are minimum, maximum, mean, median, standard deviation, variance, quartiles, range, interquartile range, root mean square, and number of peaks.

### c(ii)

In [4]:
rows = []

for activity, dataset_number, dataframe in data:
    row = {"activity": activity, "dataset": dataset_number}
    for series_number, column in enumerate(cols[1:], 1):
        row[f"min{series_number}"] = dataframe[column].min()
        row[f"max{series_number}"] = dataframe[column].max()
        row[f"mean{series_number}"] = dataframe[column].mean()
        row[f"median{series_number}"] = dataframe[column].median()
        row[f"std{series_number}"] = dataframe[column].std()
        row[f"1st_quart{series_number}"] = dataframe[column].quantile(0.25)
        row[f"3rd_quart{series_number}"] = dataframe[column].quantile(0.75)
        
    rows.append(row)

features = pd.DataFrame(rows, index=range(1, len(rows)+1))
features.index.name = "Instance"

print("Feature dataset:", features.shape[0], "instances,", features.shape[1]-2, "features")
features

Feature dataset: 88 instances, 42 features


,activity,dataset,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,bending1,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,bending1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,bending1,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,bending1,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,bending1,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,walking,11,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,walking,12,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,walking,13,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


### c(iii)

In [5]:
random_generator = np.random.default_rng(0)
results = []

for feature in features.columns[2:]:
    values = features[feature].values
    sample_indices = random_generator.integers(0, len(values), (1000, len(values)))
    bootstrap_stds = values[sample_indices].std(axis=1, ddof=1)
    results.append([feature, values.std(ddof=1), np.percentile(bootstrap_stds, 5), np.percentile(bootstrap_stds, 95)])

confidence_intervals = pd.DataFrame(results, columns=["Feature", "Standard Deviation", "90% CI Lower Bound", "90% CI Upper Bound"]).set_index("Feature")

print("Standard deviation of each feature with 90% bootstrap confidence interval:")
confidence_intervals

Standard deviation of each feature with 90% bootstrap confidence interval:


,Standard Deviation,90% CI Lower Bound,90% CI Upper Bound
Feature,,,
min1,9.569975,8.295475,10.819478
max1,4.394362,3.356211,5.259539
mean1,5.335718,4.702943,5.884677
median1,5.440054,4.806119,6.008844
std1,1.772153,1.575131,1.956231
1st_quart1,6.153590,5.578463,6.613689
3rd_quart1,5.138925,4.342492,5.852185
min2,0.000000,0.000000,0.000000
max2,5.062729,4.637455,5.404565


### c(iv)

Here, the three features selected are mean, first quartile and third quartile. Minimum was not chosen here because the minimum of the variance series min2, min4 and min6 is 0 or close to 0 in every instance, which gives a standard deviation close to 0, so it does not distinguish between activities. Maximum was excluded here as it is easily affected by single spikes in the signal. The mean reflects the overall signal level, which varies clearly across activities, while the first and third quartiles describe the spread of the central values and are robust to outliers. Also, median was not included since it is nearly identical to the mean for these series.

## Question 2

### (a)

Here, we would expect one to be lower than the other, that is, we would expect the training RSS of the cubic regression to be lower than that of the linear regression. The linear model is a special case of the cubic model with β2 = 0 and β3 = 0, so least squares on the cubic model can always do at least as well on the training data. Even though the true relationship is linear, the extra terms give the cubic model more flexibility, which lets it fit some of the random noise in the training data and reduce the RSS further.

### (b)

Here, we would expect the test RSS of the linear regression to be lower. The extra terms in the cubic model do not reduce bias as the true relationship is linear. They only add variance, since the cubic model overfits the noise in the training data, thus leading to a higher error on new observations.

### (c)

Here, we would expect the training RSS of the cubic regression to be lower. As in our answer in (a), the linear model is nested within the cubic model, so regardless of what the true relationship is, the cubic model will always fit the training data at least as well as the linear model, which means its training RSS will be lower than or equal to that of the linear model.

### (d)

In this case, there is not enough information to tell. If the true relationship here is close to linear, the linear regression may have a lower test RSS since the cubic model adds variance without much reduction in bias. If the true relationship is far from linear, the cubic regression would likely have a lower test RSS because its reduction in bias outweighs the increase in variance. This depends on the bias-variance tradeoff, and also how far the true relationship is from linear.